# Lekcija 11: Zaglađivanje i Gaussove piramide

U Lekciji 10 Gaussovo zamućivanje predstavljeno je kao jedna od konvolucijskih jezgri. Ovdje razmatramo *zašto* je zamućivanje važno i izvan samog „omekšavanja” slike: ono je ključan korak za pravilno smanjenje učestalosti uzorkovanja. To nas izravno vodi do **Gaussove piramide** &mdash; niza postupno manjih i zamućenijih inačica slike, koji se u računalnom vidu koristi za analizu na više skala.


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Zašto je prije smanjenja uzorkovanja potrebno zamućivanje

Jednostavno smanjivanje slike zadržavanjem svakog $k$-tog piksela (smanjenje uzorkovanja najbližim susjedom) može izazvati **aliasing**: fini periodični detalji koji osciliraju brže nego što ih novi razmak između piksela može prikazati pretvaraju se u potpuno drukčiji, prividan niskofrekvencijski uzorak.

To ćemo pokazati na fotografiji zida od opeke &mdash; ponavljajući redovi opeka i linije morta upravo su vrsta finih, pravilnih detalja kod kojih je aliasing posebno izražen.


In [ ]:
brick_wall = cv2.imread('../img/brick_wall.jpg', cv2.IMREAD_GRAYSCALE)

factor = 4
naive_downsample = brick_wall[::factor, ::factor]                                        # subsample directly
safe_downsample = cv2.GaussianBlur(brick_wall, (0, 0), sigmaX=factor / 2)[::factor, ::factor]  # blur first

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
axes[0].imshow(brick_wall, cmap='gray')
axes[0].set_title('Original')
axes[1].imshow(naive_downsample, cmap='gray')
axes[1].set_title('Naive: subsample only\n(aliased, noisy-looking)')
axes[2].imshow(safe_downsample, cmap='gray')
axes[2].set_title('Blur, then subsample\n(correctly soft)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

<p class="image-attribution">Izvor slike: <a href="https://www.publicdomainpictures.net/" target="_blank" rel="noopener">publicdomainpictures.net</a></p>


Jednostavna inačica pretvara pravilan uzorak opeke u sitne točkice nalik šumu &mdash; klasičan artefakt aliasinga, isti učinak zbog kojeg kotači automobila na snimci mogu izgledati kao da se okreću unatrag. Zamućena inačica ostaje prepoznatljiva kao zid od opeke jer su linije morta zaglađene *prije* poduzorkovanja, umjesto da budu proizvoljno zadržane ili odbačene piksel po piksel.


## Odabir sigme i veličine jezgre

Za pravilno zamućivanje prije smanjenja uzorkovanja potrebna su dva uvjeta: odgovarajuća vrijednost `sigma` i jezgra *dovoljno velika da je prikaže*. Parametar `sigma` funkcije `cv2.GaussianBlur` određuje raspon težina zvonolike krivulje: veća sigma usrednjava šire susjedstvo i uklanja sitnije detalje. Konkretno, zamućivanje pojedinačnog piksela Gaussovom jezgrom veličine 3 × 3 samo je **ponderirani prosjek njegova susjedstva veličine 3 × 3**: svaku od 9 vrijednosti piksela pomnožimo s odgovarajućom težinom jezgre i zbrojimo. Funkcija `cv2.getGaussianKernel(3, sigma)` daje jednodimenzionalne težine; njihov vanjski produkt daje dvodimenzionalnu jezgru.


In [ ]:
rng = np.random.default_rng(0)
noise_img = rng.integers(0, 255, (20, 20)).astype(np.uint8)  # single-channel, for a simple example

sigma = 1.0
k1d = cv2.getGaussianKernel(3, sigma)
kernel2d = k1d @ k1d.T
print('3x3 Gaussian kernel (sigma=1.0):\n', np.round(kernel2d, 3))
print('weights sum to:', kernel2d.sum())

y, x = 10, 10
neighborhood = noise_img[y - 1:y + 2, x - 1:x + 2].astype(np.float64)
manual = (kernel2d * neighborhood).sum()

blurred_full = cv2.GaussianBlur(noise_img, (3, 3), sigmaX=sigma)

print(f'\n3x3 neighborhood around pixel ({x}, {y}):\n{neighborhood}')
print(f'\nweighted average (manual): {manual:.2f}')
print(f'cv2.GaussianBlur pixel:    {blurred_full[y, x]}')

Ručno izračunat ponderirani prosjek podudara se s rezultatom funkcije `cv2.GaussianBlur` za taj piksel, uz moguću razliku zbog cjelobrojnog zaokruživanja &mdash; svaki zamućeni piksel zapravo je rezultat takvog skalarnog produkta 9 vrijednosti, ponovljenog na svakom položaju u slici. Međutim, povećavanje sigme uz zadržavanje jezgre veličine 3 × 3 gotovo ništa ne mijenja jer prozor te veličine ima mjesta za samo 3 koeficijenta po osi &mdash; da bismo vidjeli učinak šire sigme, mora se povećati i jezgra.


In [ ]:
img = np.zeros((150, 150, 3), dtype=np.uint8)
cv2.circle(img, (75, 75), 55, (255, 120, 30), -1)
cv2.rectangle(img, (20, 20), (60, 60), (30, 200, 255), -1)

sigmas = [0, 1, 3, 8]
fig, axes = plt.subplots(1, len(sigmas), figsize=(12, 3.5))
for ax, s in zip(axes, sigmas):
    # ksize=(0, 0) tells OpenCV to pick a kernel size automatically, big enough for this sigma
    blurred = img if s == 0 else cv2.GaussianBlur(img, (0, 0), sigmaX=s)
    ax.imshow(blurred)
    ax.set_title(f'sigma = {s}')
    ax.axis('off')
plt.tight_layout()
plt.show()

### Veličina jezgre mora pratiti sigmu

Parametrom `ksize=(0, 0)` u prethodnom primjeru zatražili smo da OpenCV automatski odredi veličinu jezgre za svaku sigmu. Što se događa ako umjesto toga zadamo premalu fiksnu jezgru? Funkcija `cv2.getGaussianKernel` uvijek ponovno normalizira težine tako da im zbroj bude 1, bez obzira na broj koeficijenata &mdash; pa premala jezgra ne odsijeca samo repove Gaussove funkcije, nego bez upozorenja mijenja oblik cijele jezgre u nešto nalik običnom filtru srednje vrijednosti, gubeći zvonoliki oblik.


In [ ]:
sigma = 5.0
k3 = cv2.getGaussianKernel(3, sigma).ravel()
print(f'a 3-tap kernel for sigma={sigma}, renormalized to sum to 1:', np.round(k3, 3))
print('(nearly identical to a plain 3-tap box average [0.33, 0.33, 0.33] -- the wide bell curve got squashed away)')

too_small = cv2.GaussianBlur(img, (3, 3), sigmaX=sigma)   # kernel far too small for this sigma
auto = cv2.GaussianBlur(img, (0, 0), sigmaX=sigma)         # OpenCV auto-sizes the kernel to fit sigma

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, im, title in zip(axes, [img, too_small, auto],
                          ['Original', 'ksize=3\n(too small for sigma=5)', 'ksize=0\n(auto-sized for sigma=5)']):
    ax.imshow(im)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f'mean abs difference, too-small vs. correctly-sized kernel: {np.abs(too_small.astype(int) - auto.astype(int)).mean():.2f}')

Rezultati su vidljivo različiti &mdash; `ksize=3` gotovo uopće ne zamućuje sliku unatoč traženoj vrijednosti `sigma=5`, jer prozor s 3 koeficijenta ne može prikazati tako široku zvonoliku krivulju. Upravo zato kod za piramidu u nastavku, kao i prethodno rješenje aliasinga, koristi `ksize=(0, 0)`: OpenCV tako odabire dovoljno široku jezgru za zadanu sigmu, čime se izbjegava rizik premale jezgre bez upozorenja.


## Izrada Gaussove piramide

**Gaussova piramida** ponavlja postupak „zamućivanje pa smanjenje uzorkovanja faktorom 2”, stvarajući niz slika u kojem svaka ima upola manju širinu i visinu od prethodne. Svaka razina predstavlja grublji prikaz istog prizora uz pravilno suzbijen aliasing &mdash; nije riječ samo o manjem izrezu slike.


In [ ]:
def gaussian_pyramid(image, num_levels, sigma=1.0):
    pyramid = [image]
    current = image
    for _ in range(num_levels - 1):
        blurred = cv2.GaussianBlur(current, (0, 0), sigmaX=sigma)
        current = blurred[::2, ::2]
        pyramid.append(current)
    return pyramid

photo = np.zeros((256, 256, 3), dtype=np.uint8)
photo[:] = (40, 40, 40)
cv2.circle(photo, (128, 128), 90, (255, 120, 30), -1)
cv2.rectangle(photo, (30, 30), (110, 110), (30, 200, 255), -1)

pyramid = gaussian_pyramid(photo, num_levels=5)

fig, axes = plt.subplots(1, len(pyramid), figsize=(13, 3))
for ax, level in zip(axes, pyramid):
    ax.imshow(level)
    ax.set_title(f'{level.shape[1]}x{level.shape[0]}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Usporedba s ugrađenom funkcijom `cv2.pyrDown` iz biblioteke OpenCV

OpenCV nudi funkciju `cv2.pyrDown`, koja primjenjuje istu ideju zamućivanja pa smanjenja uzorkovanja, koristeći fiksnu, pažljivo oblikovanu binomnu jezgru s 5 koeficijenata koja aproksimira Gaussovu funkciju, umjesto proizvoljne vrijednosti `sigma`. Izlazne dimenzije potpuno se podudaraju s našima; vrijednosti piksela bliske su, ali nisu identične jer se jezgre malo razlikuju.


In [ ]:
cv_pyramid = [photo]
current = photo
for _ in range(4):
    current = cv2.pyrDown(current)
    cv_pyramid.append(current)

for ours, cvs in zip(pyramid, cv_pyramid):
    assert ours.shape == cvs.shape
    diff = np.abs(ours.astype(int) - cvs.astype(int))
    print(f'{ours.shape[1]:>4}x{ours.shape[0]:<4}  mean abs diff = {diff.mean():.2f}')

## Piramide gube informacije

Smanjenje uzorkovanja nije reverzibilno: povećavanje slike s grublje razine piramide natrag na izvornu veličinu (`cv2.pyrUp`) ne može vratiti detalje odbačene zamućivanjem i poduzorkovanjem. Upravo tu razliku između povećane grublje razine i izvornika na svakoj razini pohranjuje **Laplacianova piramida**, koju ćemo obraditi u jednoj od sljedećih lekcija. Gubitak informacija već sada možemo izravno vidjeti.


In [ ]:
level1 = pyramid[1]                              # 128x128, one pyrDown from the original
reconstructed = cv2.pyrUp(level1)                 # back up to 256x256

diff = cv2.absdiff(photo, reconstructed)

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, im, title in zip(axes, [photo, reconstructed, diff],
                          ['Original', 'pyrDown then pyrUp', 'Difference (lost detail)']):
    ax.imshow(im)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f'mean absolute reconstruction error = {diff.astype(np.float64).mean():.2f} gray levels')

Oštri rubovi kruga i kvadrata nakon povećavanja postaju mekši i izgledaju pomaknuto &mdash; fini detalji trajno su odbačeni zamućivanjem i poduzorkovanjem, a `pyrUp`, odnosno interpolacija, može ih samo procijeniti, ne i obnoviti.


### Zadaci

1. Ponovite pokus sa zidom od opeke koristeći `factor = 2` umjesto `4`. Je li aliasing i dalje vidljiv pri izravnom poduzorkovanju? Zašto bi manji faktor smanjenja uzorkovanja mogao uzrokovati manje aliasinga?
2. Gaussove piramide koriste se za pretraživanje od grube prema finoj razini, primjerice za brzo određivanje približnog položaja objekta na maloj razini piramide, a zatim preciziranje na većim razinama. Zašto bi preskakanje zamućivanja i izravno poduzorkovanje narušili tu strategiju?
3. Izradite piramidu sa 6 razina za sliku nalik stvarnoj fotografiji i odredite nakon koliko razina slika postaje premala za prepoznavanje bilo kojeg oblika. Kojoj bi stvarnoj razlučivosti to odgovaralo, primjerice za fotografiju veličine 1920 × 1080?
